# 03 - Split, Drift, and Feature Engineering

**Objectif :** créer un split de développement depuis le train brut, vérifier le leakage, analyser le drift et créer les features métier sans toucher au test final.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load raw train and keep final test untouched

In [ ]:
from credit_risk_lab.infrastructure import CreditRiskQualityChecker
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

raw_train_df = CsvLoanDataLoader(path=settings.raw_train_path).load()
raw_test_path = settings.raw_test_path

clean_train_df = CreditRiskQualityChecker().clean(raw_train_df)

{
    "clean_train_rows": len(clean_train_df),
    "reserved_final_test_path": str(raw_test_path),
}

## 2. Development train/validation split

In [ ]:
from credit_risk_lab.application import DatasetSplitter, SplitConfig

split_config = SplitConfig(
    test_size=settings.validation_size,
    random_state=settings.random_state,
    stratify=True,
)
splitter = DatasetSplitter(split_config)

model_train_df, validation_df = splitter.split(clean_train_df)
split_summary = splitter.summary(model_train_df, validation_df)
split_summary

## 3. Data leakage checks before preprocessing

In [ ]:
from credit_risk_lab.infrastructure.analytics import DataLeakageAuditor

leakage_auditor = DataLeakageAuditor(target_column=settings.target_column)
overlap_report = leakage_auditor.row_overlap_report(
    model_train_df,
    validation_df,
    holdout_name="validation",
)
target_report = leakage_auditor.target_leakage_report(
    model_train_df.drop(columns=[settings.target_column])
)

display(overlap_report)
target_report

## 4. Persist development artifacts

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CSVDatasetRepository

train_path = CSVDatasetRepository.save(model_train_df, settings.train_path)
validation_path = CSVDatasetRepository.save(validation_df, settings.validation_path)

{"train_path": str(train_path), "validation_path": str(validation_path)}

## 5. Train vs validation drift analysis

In [ ]:
from credit_risk_lab.infrastructure.analytics import DriftAnalyzer

drift_analyzer = DriftAnalyzer(bins=10)
monitored_features = [
    column for column in model_train_df.columns if column != settings.target_column
]
drift_report = drift_analyzer.report_frame(
    model_train_df,
    validation_df,
    features=monitored_features,
)
drift_report.head(10)

## 6. Drift visualisation

In [ ]:
from credit_risk_lab.infrastructure.visualization import plot_drift_summary

plot_drift_summary(drift_report).show()

## 7. Business feature engineering on train only

In [ ]:
from credit_risk_lab.infrastructure.feature_engineering import LoanFeatureEngineer

feature_engineer = LoanFeatureEngineer()
featured_train_df = feature_engineer.transform(model_train_df)
created_features = [
    column for column in featured_train_df.columns if column not in model_train_df.columns
]

display(featured_train_df.head())
pd.DataFrame({"created_feature": created_features})